In [1]:
import pandas as pd
import scanpy as sc
import os
from scipy.sparse import issparse

In [2]:
results = sc.read('../predict.h5ad')

In [4]:
results.obs.to_csv('results.csv')

In [2]:
data = pd.read_csv('ig_score_changes.csv')

In [3]:
data

,Gene,IG Score Before Training,IG Score After Training
0,C1orf141,0.268941,0.000243
1,PKP1,0.268941,0.000022
2,RERE,0.268941,0.000227
3,CSMD2,0.268941,0.000242
4,HIVEP3,0.268941,0.000245
...,...,...,...
27195,ANKRD20A12P,0.268941,0.268941
27196,MAFIP,0.268941,0.268941
27197,MGC70870,0.268941,0.268941
27198,LOC102724728,0.268941,0.268941


In [5]:
data['change'] = data['IG Score Before Training'] - data['IG Score After Training']

In [6]:
data

,Gene,IG Score Before Training,IG Score After Training,change
0,C1orf141,0.268941,0.000243,0.268699
1,PKP1,0.268941,0.000022,0.268919
2,RERE,0.268941,0.000227,0.268714
3,CSMD2,0.268941,0.000242,0.268700
4,HIVEP3,0.268941,0.000245,0.268696
...,...,...,...,...
27195,ANKRD20A12P,0.268941,0.268941,0.000000
27196,MAFIP,0.268941,0.268941,0.000000
27197,MGC70870,0.268941,0.268941,0.000000
27198,LOC102724728,0.268941,0.268941,0.000000


In [7]:
#sort by change
data.sort_values('change', ascending=False)

,Gene,IG Score Before Training,IG Score After Training,change
14605,VIM,0.268941,0.000005,0.268937
17412,YBX3,0.268941,0.000014,0.268928
1840,MCL1,0.268941,0.000015,0.268927
23186,LGALS3BP,0.268941,0.000015,0.268927
25274,LENG8,0.268941,0.000015,0.268927
...,...,...,...,...
25970,WFDC2,0.268941,0.954725,-0.685784
24846,APOE,0.268941,0.973593,-0.704652
13012,TMSB4X,0.268941,0.979171,-0.710229
3724,SFTPB,0.268941,0.982532,-0.713591


In [9]:
data.to_csv('change.csv',index=False)

In [4]:
import os
def find_T_cell_h5ad(root_dir):
    T_cell_paths = []
    for dirpath, _, filenames in os.walk(root_dir):
        if 'T_cell.h5ad' in filenames:
            T_cell_paths.append(os.path.join(dirpath, 'T_cell.h5ad'))
    
    return T_cell_paths

def main():
    root_dir = '/project/DPDS/Wang_lab/shared/spatial_TCR/data/train_validate/regular_SRT/'  
    T_cell_paths = find_T_cell_h5ad(root_dir)
    
    for path in T_cell_paths:
        print(path)

if __name__ == "__main__":
    main()

/project/DPDS/Wang_lab/shared/spatial_TCR/data/train_validate/regular_SRT/Puck_200727_13/T_cell.h5ad
/project/DPDS/Wang_lab/shared/spatial_TCR/data/train_validate/regular_SRT/Puck_200727_12/T_cell.h5ad
/project/DPDS/Wang_lab/shared/spatial_TCR/data/train_validate/regular_SRT/Puck_220408_13/T_cell.h5ad
/project/DPDS/Wang_lab/shared/spatial_TCR/data/train_validate/regular_SRT/Puck_200727_10/T_cell.h5ad
/project/DPDS/Wang_lab/shared/spatial_TCR/data/train_validate/regular_SRT/Puck_200727_08/T_cell.h5ad
/project/DPDS/Wang_lab/shared/spatial_TCR/data/train_validate/regular_SRT/Puck_200727_09/T_cell.h5ad
/project/DPDS/Wang_lab/shared/spatial_TCR/data/train_validate/regular_SRT/Puck_220215_11/T_cell.h5ad
/project/DPDS/Wang_lab/shared/spatial_TCR/data/train_validate/regular_SRT/Puck_220408_09/T_cell.h5ad
/project/DPDS/Wang_lab/shared/spatial_TCR/data/train_validate/regular_SRT/Puck_220215_14/T_cell.h5ad
/project/DPDS/Wang_lab/shared/spatial_TCR/data/train_validate/regular_SRT/Puck_220408_08/T_

In [3]:
def preprocess_data(adata, immune_cell, n_genes):
    # Read the data
    if immune_cell == 'tcell':
        immune_cell = 'T'
    elif immune_cell == 'bcell':
        immune_cell = 'B'
    else:
        raise ValueError('Invalid immune cell type')

    # Ensure adata is not a view
    adata = adata.copy()

    # Filter the tumor cells
    tumor_cells = adata[adata.obs['cell_type'].astype(int) == 1].copy()

    # Check if the expression matrix is sparse and convert to dense if necessary
    if issparse(tumor_cells.X):
        tumor_cells_X_dense = tumor_cells.X.toarray()
    else:
        tumor_cells_X_dense = tumor_cells.X

    # Calculate mean expression
    mean_expression = tumor_cells_X_dense.mean(axis=0)

    # Select top n genes
    top_n_genes = mean_expression.argsort()[-n_genes:][::-1]
    adata = adata[:, top_n_genes].copy()
    adata.obs[immune_cell] = adata.obs[immune_cell].astype(float)
    tumor_cells.obs[immune_cell] = tumor_cells.obs[immune_cell].astype(float)
    # Calculate the 50th percentile of the immune cell column
    percentile_value = np.percentile(tumor_cells.obs[immune_cell], 50)

    # Binarize the immune cell column based on the percentile value
    adata.obs[immune_cell] = np.where(adata.obs[immune_cell] >= percentile_value, 1, 0)

    return adata

In [24]:
adata = sc.read_h5ad('/project/DPDS/Wang_lab/shared/spatial_TCR/data/train_validate/Spatial_TCR_Visium/p16/p16.h5ad')

In [17]:
adata.obs['T'].unique()

array([0, 1])

In [25]:
adata = sc.read_h5ad('/project/DPDS/Wang_lab/shared/spatial_TCR/data/train_validate/Visium/HumanBreastCancerWholeTranscriptome/T_cell.h5ad')

In [27]:
print("Unique cell_type values:", adata.obs['cell_type'].unique())

Unique cell_type values: ['1', '0']
Categories (2, object): ['0', '1']


In [21]:
tumor_cells = adata[adata.obs['cell_type'].astype(int) == 1].copy()

In [22]:
 if issparse(adata.X):
    tumor_cells_X_dense = tumor_cells.X.toarray()
else:
    tumor_cells_X_dense = tumor_cells.X



In [23]:
mean_expression = tumor_cells_X_dense.mean(axis=0)

In [1]:
import pandas as pd

In [4]:
df1 = pd.read_csv('data/tumor_antigens.csv')
df2 = pd.read_csv('./ig_score_changes_10000.csv')

In [5]:
df2.dropna( inplace=True)

In [6]:
df2

,Gene,IG Score Before Training,IG Score After Training,Difference
0,COL1A1,0.268941,0.996046,0.727105
1,COL3A1,0.268941,0.992967,0.724025
2,CD74,0.268941,0.992216,0.723275
3,IGFBP5,0.268941,0.992187,0.723246
4,JCHAIN,0.268941,0.991587,0.722645
...,...,...,...,...
15126,FTH1,0.268941,0.001346,-0.267595
15127,SCG2,0.268941,0.000757,-0.268185
15128,PIGR,0.268941,0.000460,-0.268481
15129,CHGA,0.268941,0.000216,-0.268725


In [12]:
common_genes = set(df1['Gene']).intersection(df2['Gene'])
df3 = pd.DataFrame(common_genes, columns=['Gene'])

In [13]:
len(common_genes)

1440

In [15]:
df3

,Gene
0,PDZD7
1,SMC1B
2,NOX4
3,MIAT
4,CDKN3
...,...
1435,HCN2
1436,NOX1
1437,MMP11
1438,CCL18


In [16]:
df3.to_csv('common_genes.csv', index=False)

In [17]:
df4= pd.read_csv('data/human.csv')
common_genes = set(df3['Gene']).intersection(df4['Gene'])

In [18]:
len(common_genes)

1440